In [5]:
pip install scipy

   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 308.4 kB/s eta 0:01:58
    --------------------------------------- 0.5/36.6 MB 308.4 kB/s eta 0:01:58
  


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Setup and Dataset (Run this in the first code cell of your Jupyter Notebook)
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
n = 40
df = pd.DataFrame({
    "Student_ID": range(1, n + 1),
    "Branch": np.random.choice(["CSE", "ECE", "ME", "CE"], n),
    "Study_Hours": np.round(np.random.uniform(1, 8, n), 1),
    "Attendance": np.random.randint(55, 100, n),
})
df["Marks"] = np.clip(
    np.round(30 + 5 * df["Study_Hours"] + 0.2 * df["Attendance"] + np.random.normal(0, 6, n)),
    0, 100
).astype(int)
df.head()

,Student_ID,Branch,Study_Hours,Attendance,Marks
0,1,ME,5.3,70,75
1,2,CE,2.0,99,68
2,3,CSE,3.0,72,59
3,4,ME,3.6,78,70
4,5,ME,4.2,80,69


In [4]:
# Task 1: Define grading function, apply it to add 'Grade' column, and show value counts
def get_grade(marks):
    if marks >= 75:
        return "A"
    elif marks >= 60:
        return "B"
    elif marks >= 40:
        return "C"
    else:
        return "F"

df["Grade"] = df["Marks"].apply(get_grade)
print(df["Grade"].value_counts())

Grade
B    16
A    12
C    12
Name: count, dtype: int64


In [7]:
# Task 2: List comprehension, filter with lambda, and map with lambda for grace marks
low_att_marks = [m for m, att in zip(df["Marks"], df["Attendance"]) if att < 75]
print("Marks with attendance below 75:", low_att_marks)

above_70 = list(filter(lambda x: x > 70, df["Marks"]))
print("Marks above 70:", above_70)

grace_marks = list(map(lambda x: min(x + 5, 100), df["Marks"]))
print("Marks after 5 grace marks:", grace_marks)

Marks with attendance below 75: [75, 59, 60, 57, 78, 82, 55, 80, 48, 64, 54, 58, 65, 82, 80, 69]
Marks above 70: [75, 74, 81, 72, 78, 82, 80, 87, 78, 78, 83, 82, 80, 84]
Marks after 5 grace marks: [80, 73, 64, 75, 74, 79, 69, 86, 77, 65, 66, 62, 59, 83, 87, 72, 60, 65, 85, 68, 53, 69, 59, 92, 64, 83, 63, 83, 73, 64, 88, 70, 87, 85, 74, 89, 52, 60, 55, 65]


In [8]:
# Task 3: Flexible summary function and testing
def summary(*values, round_to=2):
    vals = list(values)
    return {
        "count": len(vals),
        "minimum": round(min(vals), round_to) if vals else 0,
        "maximum": round(max(vals), round_to) if vals else 0,
        "average": round(sum(vals) / len(vals), round_to) if vals else 0
    }

print("Summary of 10, 20, 30:", summary(10, 20, 30))
print("Summary of df Marks:", summary(*df["Marks"]))

Summary of 10, 20, 30: {'count': 3, 'minimum': 10, 'maximum': 30, 'average': 20.0}
Summary of df Marks: {'count': 40, 'minimum': 47, 'maximum': 87, 'average': 66.92}


In [9]:
# Task 4: Save dataframe to CSV and write safe loading function with try-except
df.to_csv("students.csv", index=False)

def load_data(filename):
    try:
        data = pd.read_csv(filename)
        print(f"Successfully loaded '{filename}'.")
        return data
    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found. Please check the file name.")

# Test with correct file name
load_data("students.csv")

# Test with incorrect file name
load_data("wrong_students.csv")

Successfully loaded 'students.csv'.
Error: The file 'wrong_students.csv' was not found. Please check the file name.


In [10]:
# Task 5: Group by branch, find highest average marks, and show top 5 students
branch_stats = df.groupby("Branch")[["Marks", "Study_Hours"]].mean()
print(branch_stats)

highest_branch = branch_stats["Marks"].idxmax()
print(f"\nBranch with the highest average marks: {highest_branch}")

top_5_students = df.sort_values(by="Marks", ascending=False).head(5)
print("\nTop 5 students by marks:")
print(top_5_students)

            Marks  Study_Hours
Branch                        
CE      68.416667       4.4500
CSE     67.200000       4.6600
ECE     65.625000       3.7875
ME      65.900000       4.2200

Branch with the highest average marks: CE

Top 5 students by marks:
    Student_ID Branch  Study_Hours  Attendance  Marks
23          24     CE          7.4          89     87
35          36    CSE          7.5          88     84
30          31    CSE          7.8          82     83
14          15     CE          7.8          63     82
32          33     CE          7.6          63     82


In [11]:
# Task 6: Calculate z-scores using NumPy without loops and count those > 1
marks_array = df["Marks"].to_numpy()
z_scores = (marks_array - np.mean(marks_array)) / np.std(marks_array)
print("Z-scores:", z_scores)

count_above_1 = np.sum(z_scores > 1)
print(f"Number of students with a z-score greater than 1: {count_above_1}")

Z-scores: [ 0.74346049  0.09897462 -0.72965008  0.28311344  0.19104403  0.65139108
 -0.26930302  1.29587695  0.46725226 -0.63758067 -0.54551126 -0.9137889
 -1.18999713  1.01966872  1.38794636  0.00690521 -1.09792772 -0.63758067
  1.20380754 -0.36137243 -1.74241359 -0.26930302 -1.18999713  1.84829341
 -0.72965008  1.01966872 -0.82171949  1.01966872  0.09897462 -0.72965008
  1.48001577 -0.17723361  1.38794636  1.20380754  0.19104403  1.57208518
 -1.834483   -1.09792772 -1.55827477 -0.63758067]
Number of students with a z-score greater than 1: 11


In [12]:
# Task 7: Descriptive statistics and outlier detection using IQR
marks = df["Marks"]
mean_val = marks.mean()
median_val = marks.median()
mode_val = marks.mode()[0]
var_val = marks.var()
std_val = marks.std()
min_val = marks.min()
max_val = marks.max()

Q1 = marks.quantile(0.25)
Q3 = marks.quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print(f"Mean: {mean_val:.2f}, Median: {median_val}, Mode: {mode_val}, Variance: {var_val:.2f}, Std Dev: {std_val:.2f}, Min: {min_val}, Max: {max_val}")
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Outlier limits: Lower = {lower_limit}, Upper = {upper_limit}")

Mean: 66.92, Median: 66.0, Mode: 59, Variance: 120.99, Std Dev: 11.00, Min: 47, Max: 87
Q1: 59.0, Q3: 78.0, IQR: 19.0
Outlier limits: Lower = 30.5, Upper = 106.5


In [13]:
# Task 8: Correlation of Marks with Study_Hours and Attendance
correlation_matrix = df[["Marks", "Study_Hours", "Attendance"]].corr()
print(correlation_matrix)

                Marks  Study_Hours  Attendance
Marks        1.000000     0.853736    0.066744
Study_Hours  0.853736     1.000000   -0.143288
Attendance   0.066744    -0.143288    1.000000


We are 95% confident that the true population average marks for the students lie within this interval range.

In [14]:
# Task 9: 95% confidence interval for average marks
mean_marks = df["Marks"].mean()
sem_marks = stats.sem(df["Marks"])
conf_interval = stats.t.interval(0.95, len(df["Marks"]) - 1, loc=mean_marks, scale=sem_marks)
print("95% Confidence Interval:", conf_interval)

95% Confidence Interval: (np.float64(63.407113198283106), np.float64(70.4428868017169))


$H_0$: The true average marks of students is equal to 60 ($\mu = 60$).$H_1$: The true average marks of students is not equal to 60 ($\mu \neq 60$).
At a significance level of 0.05, if the p-value is less than 0.05, we reject $H_0$, meaning the department's claim of an average of 60 is statistically unlikely. If it is greater, we fail to reject $H_0$.

In [15]:
# Task 10: One-sample t-test against a population mean of 60
t_stat, p_val = stats.ttest_1samp(df["Marks"], 60)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4f}")

T-statistic: 3.9817, P-value: 0.0003


$H_0$: There is no significant difference in average marks between CSE and ECE students.
$H_1$: There is a significant difference in average marks between CSE and ECE students.
Based on the resulting p-value compared against the 0.05 threshold, we either conclude that there is a statistically significant difference in performance between the two branches or that the difference could be due to random chance.

In [16]:
# Task 11: Independent t-test comparing CSE and ECE marks
cse_marks = df[df["Branch"] == "CSE"]["Marks"]
ece_marks = df[df["Branch"] == "ECE"]["Marks"]
t_stat_ind, p_val_ind = stats.ttest_ind(cse_marks, ece_marks)
print(f"Independent T-statistic: {t_stat_ind:.4f}, P-value: {p_val_ind:.4f}")

Independent T-statistic: 0.2758, P-value: 0.7863


The analysis indicates that student performance varies moderately across the class, with self-study hours emerging as the primary factor driving higher marks. Attendance shows a positive association, but study commitment has a much stronger impact on final scores. Statistical testing helps validate group performance variations and gives reliable estimates of overall academic trends.